# 04_shap.ipynb
**Purpose:** SHAP explainability analysis — feature importance bar plots and beeswarm plots for all tree-based models.

| | |
|---|---|
| **Input** | Trained models + `X`, `X_test` from `03_model.ipynb` (re-fit here from `data/analysis_clean.csv`) |
| **Output** | `output/fig_shap_xgb_bar.png`, `output/fig_shap_xgb_beeswarm.png`, `output/fig_shap_cat_bar.png`, `output/fig_shap_cat_beeswarm.png`, `output/fig_shap_lgbm_bar.png`, `output/fig_shap_lgbm_beeswarm.png`, `output/fig_shap_gb_bar.png` |
| **Key functions** | `load_and_prep()`, `refit_models()`, `save_fig()`, `shap_bar()`, `shap_beeswarm()` |

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from pathlib import Path

In [2]:
#Paths
ROOT       = Path.cwd().parent
DATA_DIR   = ROOT / "data"
OUTPUT_DIR = ROOT / "output"

INPUT_FILE  = DATA_DIR / "analysis_clean.csv"   # adjust per notebook
OUTPUT_FILE = DATA_DIR / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_COL   = 'hourly_wage'
RANDOM_STATE = 42

VARS = {
    'WF9_WORK_WAGE':              'hourly_wage',
    'WF9_CHAR_EDUC':              'education',
    'WF9_CHAR_GENDER':            'gender',
    'WF9_WORK_HRS_CAT':           'hours_cat',
    'WF9_CAREER_EXPERIENCE':      'experience',
    'WF9_CESD7_TOT':              'depression_score',
    'WF9_CHAR_HHINCOME':          'hh_income',
    'WF9_CHAR_YEAR_BORN':         'year_born',
    'WF9_REGION':                 'region',
    'WF9_WORK_FT':                'full_time_status',
    'WF9_WORK_ROLE':              'job_role',
    'WF9_WORK_MONTHS':            'months_per_year',
    'WF9_CHAR_RACE':              'race',
    'WF9_CHAR_HISP':              'hispanic_latino',
    'WF9_CHAR_COUNTRY_BORN':      'country_of_birth',
    'WF9_METH_PUBSCHFLAG':        'works_public_school',
    'WF9_CB_COMM_URBAN_DENSITY':  'urban_density',
    'WF9_PROFDEV_COURSE':         'college_for_pd',
    'WF9_PROFDEV_WRKSHP':         'workshop_for_pd',
    'WF9_A12_COLLENROLL':         'enrolled_in_college',
    'WF9_INTERACTIONS_TOT':       'interaction_quality_total',
    'WF9_E20_SRHEALTH':           'self_rated_health',
}

FEATURE_COLS = [v for v in VARS.values() if v != TARGET_COL]

In [7]:
# Functions 
def load_and_prep(path: Path, feature_cols: list, target_col: str):
    """Load clean CSV and return X, y, X_train, X_test, y_train, y_test."""
    df = pd.read_csv(path)
    print(f"[load_and_prep] Shape: {df.shape}")

    model_df = df.copy()

    model_df[target_col] = pd.to_numeric(model_df[target_col], errors='coerce')
    before = len(model_df)
    model_df = model_df.dropna(subset=[target_col])
    print(f"[load_and_prep] Dropped {before - len(model_df)} rows with non-numeric target")

    X = model_df[feature_cols].copy()
    y = model_df[target_col]

    for col in X.select_dtypes(include=['object', 'category']).columns:
        X[col] = pd.Categorical(X[col]).codes

    X = X.astype(float)

    # Step 4: drop rows with any remaining NaN — before the split
    combined = X.copy()
    combined['__target__'] = y.values
    combined = combined.dropna()
    X = combined.drop(columns='__target__')
    y = combined['__target__']
    print(f"[load_and_prep] After dropping NaN rows: {X.shape}")

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_STATE
    )
    print(f"[load_and_prep] Train: {X_train.shape}, Test: {X_test.shape}")
    return X, y, X_train, X_test, y_train, y_test

def refit_models(X_train, y_train):
    """Re-fit all four tree models with same hyperparameters as 03_model."""
    print("Fitting GradientBoosting...")
    gb = GradientBoostingRegressor(n_estimators=300, learning_rate=0.05,
                                   max_depth=4, random_state=42)
    gb.fit(X_train, y_train)

    print("Fitting LightGBM...")
    lgbm = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.3,
                              num_leaves=31, random_state=42, verbose=-1)
    lgbm.fit(X_train, y_train)

    print("Fitting XGBoost...")
    xgbm = xgb.XGBRegressor(n_estimators=500, learning_rate=0.03,
                             max_depth=4, random_state=42, verbosity=0)
    xgbm.fit(X_train, y_train)

    print("Fitting CatBoost...")
    cat = CatBoostRegressor(iterations=500, learning_rate=0.05,
                            depth=4, random_seed=42, verbose=0)
    cat.fit(X_train, y_train)

    print("[refit_models] All models fitted.")
    return gb, lgbm, xgbm, cat

def save_fig(fig: plt.Figure, filename: str) -> None:
    """Save figure to output dir and close it."""
    dest = OUTPUT_DIR / filename
    fig.savefig(dest, bbox_inches='tight', dpi=150)
    plt.close(fig)
    print(f"[save_fig] Saved → {dest}")

def shap_bar(model, data: pd.DataFrame, feature_cols: list,
             title: str, filename: str):
    """SHAP bar plot (mean absolute SHAP — feature importance)."""
    explainer = shap.TreeExplainer(model)
    shap_vals = explainer.shap_values(data)
    shap.summary_plot(shap_vals, data, feature_names=feature_cols,
                      plot_type='bar', show=False)
    plt.title(title)
    plt.tight_layout()
    save_fig(plt.gcf(), filename)
    return shap_vals

def shap_beeswarm(shap_vals, data: pd.DataFrame, feature_cols: list,
                  title: str, filename: str) -> None:
    """SHAP beeswarm plot (direction + magnitude per observation)."""
    shap.summary_plot(shap_vals, data, feature_names=feature_cols, show=False)
    plt.title(title)
    plt.tight_layout()
    save_fig(plt.gcf(), filename)

In [8]:
# Load data and refit models
X, y, X_train, X_test, y_train, y_test = load_and_prep(INPUT_FILE, FEATURE_COLS, TARGET_COL)
gb, lgbm, xgbm, cat = refit_models(X_train, y_train)

[load_and_prep] Shape: (5192, 22)
[load_and_prep] Dropped 922 rows with non-numeric target
[load_and_prep] After dropping NaN rows: (4270, 21)
[load_and_prep] Train: (3416, 21), Test: (854, 21)
Fitting GradientBoosting...
Fitting LightGBM...
Fitting XGBoost...
Fitting CatBoost...
[refit_models] All models fitted.


In [9]:
# XGBoost SHAP
print("── XGBoost SHAP ──")
xgb_shap_vals = shap_bar(xgbm, X, FEATURE_COLS,
                          'XGBoost — SHAP Feature Importance',
                          'fig_shap_xgb_bar.png')
shap_beeswarm(xgb_shap_vals, X, FEATURE_COLS,
              'XGBoost — SHAP Beeswarm',
              'fig_shap_xgb_beeswarm.png')# XGBoost SHAP
print("── XGBoost SHAP ──")
xgb_shap_vals = shap_bar(xgbm, X, FEATURE_COLS,
                          'XGBoost — SHAP Feature Importance',
                          'fig_shap_xgb_bar.png')
shap_beeswarm(xgb_shap_vals, X, FEATURE_COLS,
              'XGBoost — SHAP Beeswarm',
              'fig_shap_xgb_beeswarm.png')

── XGBoost SHAP ──
[save_fig] Saved → C:\Wage Project\output\fig_shap_xgb_bar.png
[save_fig] Saved → C:\Wage Project\output\fig_shap_xgb_beeswarm.png
── XGBoost SHAP ──
[save_fig] Saved → C:\Wage Project\output\fig_shap_xgb_bar.png
[save_fig] Saved → C:\Wage Project\output\fig_shap_xgb_beeswarm.png


In [10]:
# CatBoost SHAP
print("── CatBoost SHAP ──")
cat_shap_vals = shap_bar(cat, X_test, FEATURE_COLS,
                          'CatBoost — SHAP Feature Importance',
                          'fig_shap_cat_bar.png')
shap_beeswarm(cat_shap_vals, X_test, FEATURE_COLS,
              'CatBoost — SHAP Beeswarm',
              'fig_shap_cat_beeswarm.png')

── CatBoost SHAP ──
[save_fig] Saved → C:\Wage Project\output\fig_shap_cat_bar.png
[save_fig] Saved → C:\Wage Project\output\fig_shap_cat_beeswarm.png


In [11]:
# LightGBM SHAP
print("── LightGBM SHAP ──")
lgb_shap_vals = shap_bar(lgbm, X, FEATURE_COLS,
                          'LightGBM — SHAP Feature Importance',
                          'fig_shap_lgbm_bar.png')
shap_beeswarm(lgb_shap_vals, X, FEATURE_COLS,
              'LightGBM — SHAP Beeswarm',
              'fig_shap_lgbm_beeswarm.png')

── LightGBM SHAP ──
[save_fig] Saved → C:\Wage Project\output\fig_shap_lgbm_bar.png
[save_fig] Saved → C:\Wage Project\output\fig_shap_lgbm_beeswarm.png


In [12]:
# GradientBoosting SHAP
print("── GradientBoosting SHAP ──")
gb_shap_vals = shap_bar(gb, X_test, FEATURE_COLS,
                         'GradientBoosting — SHAP Feature Importance',
                         'fig_shap_gb_bar.png')

print("\n── AFTER SHAP ── all figures written to output/")

── GradientBoosting SHAP ──
[save_fig] Saved → C:\Wage Project\output\fig_shap_gb_bar.png

── AFTER SHAP ── all figures written to output/
